# SOTA Evaluation for Jailbreak Detection

Ноутбук для запуска SOTA-моделей на WildJailbreak test set.
Предназначен для Google Colab / Kaggle с GPU.

**Модели:**
1. PromptGuard 2 (86M) — BERT-style classifier
2. Qwen3Guard-Gen (4B / 8B) — Generative LLM

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q transformers torch datasets tqdm

In [ ]:
# Clone repository (if running in Colab)
# !git clone https://github.com/YOUR_USERNAME/autoguardrails.git
# %cd autoguardrails

# Or mount Google Drive (if repo is there)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/autoguardrails

In [ ]:
import sys
from pathlib import Path

# Add project paths
# Adjust this path based on your setup
PROJECT_ROOT = Path(".").resolve().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

TASK_DIR = PROJECT_ROOT / "tasks" / "jailbreak_detection"
DATA_DIR = TASK_DIR / "data" / "processed"
RESULTS_DIR = TASK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Task dir: {TASK_DIR}")

In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"Total memory: {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("WARNING: No GPU available!")

## 2. PromptGuard 2 Evaluation

In [ ]:
# Option 1: Run via script
# !python {TASK_DIR}/scripts/run_promptguard2.py \
#     --eval_binary {DATA_DIR}/wildjailbreak_eval_binary.jsonl \
#     --output_dir {RESULTS_DIR} \
#     --batch_size 32 \
#     --device cuda

In [ ]:
# Option 2: Run inline
import json
import numpy as np
from datetime import datetime, timezone
from tqdm import tqdm
from transformers import pipeline as hf_pipeline, AutoTokenizer

# Load data
eval_binary_path = DATA_DIR / "wildjailbreak_eval_binary.jsonl"
eval_data = []
with open(eval_binary_path, "r") as f:
    for line in f:
        eval_data.append(json.loads(line))

prompts = [r["prompt"] for r in eval_data]
y_true = np.array([1 if r["binary_label"] == "jailbreak" else 0 for r in eval_data])
data_types = np.array([r["data_type"] for r in eval_data])

print(f"Loaded {len(prompts)} examples")
print(f"  safe: {sum(y_true == 0)}, jailbreak: {sum(y_true == 1)}")

In [ ]:
# Load PromptGuard 2
model_name = "meta-llama/Llama-Prompt-Guard-2-86M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
classifier = hf_pipeline(
    "text-classification",
    model=model_name,
    tokenizer=tokenizer,
    device=0,  # GPU
)

# Check available labels
print(f"Model labels: {classifier.model.config.id2label}")

In [ ]:
# Run inference (simplified, without chunking for speed)
# For production, use the full script with chunking

from shared.metrics import evaluate_jailbreak

predictions = []
batch_size = 32

for i in tqdm(range(0, len(prompts), batch_size), desc="PromptGuard 2"):
    batch = prompts[i:i + batch_size]
    # Truncate to 512 tokens (model limit)
    batch_truncated = [p[:2000] for p in batch]  # Rough char limit
    results = classifier(batch_truncated, truncation=True, max_length=512)
    
    for r in results:
        # Adjust based on actual label names
        pred = 1 if "JAILBREAK" in r["label"].upper() else 0
        predictions.append(pred)

y_pred = np.array(predictions)

# Compute metrics
metrics = evaluate_jailbreak(y_true, y_pred, data_types, oos_label=1)
print("\nPromptGuard 2 Results:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

## 3. Qwen3Guard Evaluation

In [ ]:
# Select model size
# 4B requires ~8GB GPU RAM
# 8B requires ~16GB GPU RAM
MODEL_SIZE = "4b"  # or "8b"

In [ ]:
# Check if enough GPU memory
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    required = 8 if MODEL_SIZE == "4b" else 16
    print(f"Available: {total_mem:.1f} GB, Required: ~{required} GB")
    if total_mem < required:
        print("WARNING: May not have enough GPU memory!")

In [ ]:
# Option 1: Run via script
# !python {TASK_DIR}/scripts/run_qwen3guard.py \
#     --eval_binary {DATA_DIR}/wildjailbreak_eval_binary.jsonl \
#     --output_dir {RESULTS_DIR} \
#     --model_size {MODEL_SIZE} \
#     --batch_size 8 \
#     --device cuda

In [ ]:
# Option 2: Run inline (placeholder)
# from transformers import AutoModelForCausalLM, AutoTokenizer
#
# model_name = f"Qwen/Qwen3Guard-Gen-{MODEL_SIZE.upper()}"
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     trust_remote_code=True,
# )
#
# # ... inference code ...
print("Qwen3Guard evaluation - run the script for full implementation")

## 4. Results Comparison

In [ ]:
import pandas as pd

# Load metrics.json
metrics_path = RESULTS_DIR / "metrics.json"

if metrics_path.exists():
    with open(metrics_path, "r") as f:
        all_metrics = json.load(f)
    
    # Convert to DataFrame
    rows = []
    for model_name, metrics in all_metrics.items():
        rows.append({
            "model": model_name,
            "f1": metrics.get("f1"),
            "precision": metrics.get("precision"),
            "recall": metrics.get("recall"),
            "over_refusal_rate": metrics.get("over_refusal_rate"),
            "recall_adversarial_harmful": metrics.get("recall_adversarial_harmful"),
        })
    
    df = pd.DataFrame(rows)
    display(df)
else:
    print(f"No results found at {metrics_path}")
    print("Run the evaluation scripts first.")

In [ ]:
# Visualization (if results available)
if metrics_path.exists():
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = range(len(df))
    width = 0.2
    
    ax.bar([i - width for i in x], df["f1"], width, label="F1")
    ax.bar([i for i in x], df["precision"], width, label="Precision")
    ax.bar([i + width for i in x], df["recall"], width, label="Recall")
    
    ax.set_xlabel("Model")
    ax.set_ylabel("Score")
    ax.set_title("Jailbreak Detection: SOTA Comparison")
    ax.set_xticks(x)
    ax.set_xticklabels(df["model"], rotation=45, ha="right")
    ax.legend()
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()